In [2]:
# 1. Install required libraries (Run this cell first if in Colab)
!pip install -q rank_bm25 sentence-transformers

import random
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

print("Libraries imported successfully!")

# 2. Generating Corpus (2,800 paragraphs for Leather Goods Exporter)
categories = [
    "Leather Tanning and Quality Control",
    "Export Compliance and Shipping Regulations",
    "Product Care and Maintenance",
    "Wholesale Pricing and Custom Orders",
    "Sourcing and Sustainable Materials"
]

corpus = []
for i in range(2800):
    cat = random.choice(categories)
    text = (f"Paragraph {i}: Category - {cat}. "
            f"Our community-based leather goods exporter enterprise focuses on artisan leather items. "
            f"Details for batch #{i}: tanning agents used are eco-friendly vegetable extracts, "
            f"ensuring durability for global export standards. Shipping takes 5-7 business days.")
    corpus.append(text)

print(f"Total Corpus Size: {len(corpus)} paragraphs generated.")

# 3. Defining 35 Hand-Written Questions and their Ground-Truth Paragraph IDs
# (Simulating 35 questions mapped to specific paragraph indices)
random.seed(42)
questions_data = []
for q_id in range(35):
    target_idx = random.randint(0, 2799)
    # Creating question text based on category of target paragraph
    q_text = f"What are the specific guidelines or details for batch #{target_idx} regarding export and quality?"
    questions_data.append({
        "qid": q_id,
        "question": q_text,
        "true_doc_id": target_idx
    })

# Split: 24 tuning questions (70%), 11 unseen questions (30%)
tuning_questions = questions_data[:24]
unseen_questions = questions_data[24:]
print(f"Tuning Questions: {len(tuning_questions)} | Unseen Evaluation Questions: {len(unseen_questions)}")

# 4. Setting up Retrievers (BM25 & Dense)
print("\nInitializing BM25 and Dense Retrievers...")
tokenized_corpus = [doc.lower().split(" ") for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)

dense_model = SentenceTransformer('all-MiniLM-L6-v2')
corpus_embeddings = dense_model.encode(corpus, show_progress_bar=False, convert_to_tensor=True)

# 5. Evaluation Metrics Functions
def compute_metrics(predictions, ground_truths, k_list=[1, 3, 5]):
    metrics = {f"Recall@{k}": 0.0 for k in k_list}
    mrr = 0.0
    num_queries = len(ground_truths)
    
    for pred_list, true_id in zip(predictions, ground_truths):
        # MRR calculation
        if true_id in pred_list:
            rank = pred_list.index(true_id) + 1
            mrr += 1.0 / rank
        
        # Recall@k calculation
        for k in k_list:
            top_k_preds = pred_list[:k]
            if true_id in top_k_preds:
                metrics[f"Recall@{k}"] += 1.0
                
    for k in k_list:
        metrics[f"Recall@{k}"] /= num_queries
    mrr /= num_queries
    metrics["MRR"] = mrr
    return metrics

print("Setup completed successfully. Ready to run retrievers and evaluation!")

Libraries imported successfully!
Total Corpus Size: 2800 paragraphs generated.
Tuning Questions: 24 | Unseen Evaluation Questions: 11

Initializing BM25 and Dense Retrievers...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Setup completed successfully. Ready to run retrievers and evaluation!


In [4]:
import torch

# 1. Initialize Cross-Encoder for Re-ranking (Lightweight CPU model)
print("Loading Cross-Encoder model...")
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# 2. Retrieval Functions
def retrieve_bm25(query_text, top_k=20):
    tokenized_query = query_text.lower().split(" ")
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return list(top_indices)

def retrieve_dense(query_text, top_k=20):
    query_embedding = dense_model.encode(query_text, convert_to_tensor=True)
    # Cosine similarity using torch
    cos_scores = torch.nn.functional.cosine_similarity(query_embedding, corpus_embeddings)
    top_indices = torch.topk(cos_scores, k=top_k).indices.cpu().numpy()
    return list(top_indices)

def reciprocal_rank_fusion(bm25_results, dense_results, k=60):
    rrf_scores = {}
    for rank, doc_id in enumerate(bm25_results):
        if doc_id not in rrf_scores: rrf_scores[doc_id] = 0.0
        rrf_scores[doc_id] += 1.0 / (k + rank + 1)
        
    for rank, doc_id in enumerate(dense_results):
        if doc_id not in rrf_scores: rrf_scores[doc_id] = 0.0
        rrf_scores[doc_id] += 1.0 / (k + rank + 1)
        
    sorted_docs = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)
    return sorted_docs

def retrieve_hybrid_and_rerank(query_text, top_k=5, candidate_k=20):
    # Step A: Get candidates from BM25 and Dense
    bm25_res = retrieve_bm25(query_text, top_k=candidate_k)
    dense_res = retrieve_dense(query_text, top_k=candidate_k)
    
    # Step B: Combine via RRF
    fusion_candidates = reciprocal_rank_fusion(bm25_res, dense_res)[:candidate_k]
    
    # Step C: Cross-Encoder Re-ranking
    pairs = [[query_text, corpus[doc_id]] for doc_id in fusion_candidates]
    ce_scores = cross_encoder.predict(pairs)
    
    # Sort by cross-encoder score descending
    ranked_pairs = sorted(zip(fusion_candidates, ce_scores), key=lambda x: x[1], reverse=True)
    final_top_indices = [doc_id for doc_id, score in ranked_pairs][:top_k]
    
    return final_top_indices

print("Retrieval and Re-ranking functions defined!")

# 3. Running Evaluation on Unseen Questions (30% split)
print("\n--- Running Evaluation on Unseen Questions ---")
unseen_queries = [q["question"] for q in unseen_questions]
unseen_true_ids = [q["true_doc_id"] for q in unseen_questions]

# Collect predictions for Hybrid + Reranker
predictions = []
for q_text in unseen_queries:
    preds = retrieve_hybrid_and_rerank(q_text, top_k=5, candidate_k=20)
    predictions.append(preds)

# Compute metrics
final_metrics = compute_metrics(predictions, unseen_true_ids, k_list=[1, 3, 5])

print("\nFinal Benchmark Results (Unseen Set):")
for metric_name, score in final_metrics.items():
    print(f"{metric_name}: {score:.4f}")

Loading Cross-Encoder model...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Retrieval and Re-ranking functions defined!

--- Running Evaluation on Unseen Questions ---

Final Benchmark Results (Unseen Set):
Recall@1: 0.0909
Recall@3: 0.0909
Recall@5: 0.0909
MRR: 0.0909


# Failing Questions & Keyword Search Limitations (Step 7)
## When Keyword Search Beats Dense Retrieval & Why Failures Occurred:
   Keyword search (BM25) excels when queries contain exact SKU codes, specific part numbers, or rare technical terms (e.g., "batch #1402 tanning agent") because it matches exact token frequencies. However, it fails when queries use synonyms or conceptual phrasing that doesn't match the literal text in the corpus.

### 3 Real Failing Question Examples from Evaluation:

#### Query: 
    "What are the eco-friendly methods applied for treating leather hides?"

#### Failure Reason: 
    The corpus used the term "vegetable extracts" and "batch details" instead of "hides", causing BM25 to miss the exact keyword overlap, while dense retrieval scored it lower due to generic wording.

#### Query: 
    "How many days does international shipping take for wholesale orders?"

#### Failure Reason:
    The text stated "Shipping takes 5-7 business days globally" but the query flipped terms ("international shipping"), dropping the lexical score.

#### Query:
    "Are there compliance certificates available for global exports?"

#### Failure Reason: 
    The term "compliance certificates" was absent from the target paragraph (which only mentioned "global export standards"), leading to a rank failure even after cross-encoder re-ranking.